In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#!pip install datasets sentence-transformers scikit-learn pandas numpy


In [4]:
import os

# ✅ Fix: Centralized, OS-independent data folder (no more "../data" path issues)
DATA_DIR = "/content/drive/MyDrive/research_paper_search/data"
os.makedirs(DATA_DIR, exist_ok=True)

CLEANED_CSV_PATH = os.path.join(DATA_DIR, "cleaned_arxiv_papers.csv")
EMBEDDINGS_PATH  = os.path.join(DATA_DIR, "arxiv_embeddings.npy")

print("Data will be saved in:", DATA_DIR)


Data will be saved in: /content/drive/MyDrive/research_paper_search/data


In [5]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers")
print(dataset)


README.md:   0%|          | 0.00/986 [00:00<?, ?B/s]

ML-Arxiv-Papers.csv:   0%|          | 0.00/147M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/117592 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract'],
        num_rows: 117592
    })
})


In [6]:
import pandas as pd

df = dataset["train"].to_pandas()
print(df.columns.tolist())
df.head()


['Unnamed: 0.1', 'Unnamed: 0', 'title', 'abstract']


,Unnamed: 0.1,Unnamed: 0,title,abstract
0,0,0.0,Learning from compressed observations,The problem of statistical learning is to co...
1,1,1.0,Sensor Networks with Random Links: Topology De...,"In a sensor network, in practice, the commun..."
2,2,2.0,The on-line shortest path problem under partia...,The on-line shortest path problem is conside...
3,3,3.0,A neural network approach to ordinal regression,Ordinal regression is an important type of l...
4,4,4.0,Parametric Learning and Monte Carlo Optimization,This paper uncovers and explores the close r...


In [7]:
df = df[["title", "abstract"]]


df = df.dropna(subset=["title", "abstract"])

df = df.drop_duplicates(subset=["title"]).reset_index(drop=True)


df = df.head(50000).reset_index(drop=True)

print("Final shape after cleaning:", df.shape)
df.isnull().sum()


Final shape after cleaning: (50000, 2)


,0
title,0
abstract,0


In [8]:
df["paper_text"] = df["title"] + " " + df["abstract"]

# Clean newlines/extra spaces
df["paper_text"] = df["paper_text"].str.replace("\n", " ", regex=False)
df["paper_text"] = df["paper_text"].str.replace(r"\s+", " ", regex=True)  # ✅ Fix: collapse multiple spaces too
df["paper_text"] = df["paper_text"].str.strip()

print(df["paper_text"].iloc[0])


Learning from compressed observations The problem of statistical learning is to construct a predictor of a random variable $Y$ as a function of a related random variable $X$ on the basis of an i.i.d. training sample from the joint distribution of $(X,Y)$. Allowable predictors are drawn from some specified class, and the goal is to approach asymptotically the performance (expected loss) of the best predictor in the class. We consider the setting in which one has perfect observation of the $X$-part of the sample, while the $Y$-part has to be communicated at some finite bit rate. The encoding of the $Y$-values is allowed to depend on the $X$-values. Under suitable regularity conditions on the admissible predictors, the underlying family of probability distributions and the loss function, we give an information-theoretic characterization of achievable predictor performance in terms of conditional distortion-rate functions. The ideas are illustrated on the example of nonparametric regressio

In [9]:
from sentence_transformers import SentenceTransformer
import torch

# ✅ Fix: Auto-detect GPU, fallback to CPU (no crash on machines without GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
print(type(model))


Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>


In [10]:
sample_texts = df["paper_text"].head(5).to_list()
sample_embeddings = model.encode(sample_texts)
print("Shape:", sample_embeddings.shape)  # (5, 384)


Shape: (5, 384)


In [11]:
from sklearn.metrics.pairwise import cosine_similarity

for i in range(1, len(sample_embeddings)):
    sim = cosine_similarity(
        sample_embeddings[0].reshape(1, -1),
        sample_embeddings[i].reshape(1, -1)
    )
    print(f"Similarity Paper 0 vs Paper {i}: {sim[0][0]:.4f}")


Similarity Paper 0 vs Paper 1: 0.3663
Similarity Paper 0 vs Paper 2: 0.3352
Similarity Paper 0 vs Paper 3: 0.1551
Similarity Paper 0 vs Paper 4: 0.3742


In [12]:
import numpy as np

if os.path.exists(EMBEDDINGS_PATH):
    print("Loading saved embeddings...")
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    print("Generating embeddings for", len(df), "papers... (this may take a while)")
    embeddings = model.encode(
        df["paper_text"].tolist(),
        batch_size=32,
        show_progress_bar=True
    )
    np.save(EMBEDDINGS_PATH, embeddings)
    print("Embeddings saved to:", EMBEDDINGS_PATH)

print("Final embeddings shape:", embeddings.shape)


Generating embeddings for 50000 papers... (this may take a while)


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Embeddings saved to: /content/drive/MyDrive/research_paper_search/data/arxiv_embeddings.npy
Final embeddings shape: (50000, 384)


In [13]:
df.to_csv(CLEANED_CSV_PATH, index=False)
print("Cleaned DataFrame saved to:", CLEANED_CSV_PATH)


Cleaned DataFrame saved to: /content/drive/MyDrive/research_paper_search/data/cleaned_arxiv_papers.csv
